In [1]:
import pandas as pd 
import numpy as np 

In [ ]:
df = pd.read_csv("data/train_data.csv")
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_renovated,lat,long,hous_val_amt,trend,trend_sq,house_age,was_renovated
0,7129300520,2014-10-13,221900.0,3,1,1180,5650,1.0,0,0,...,1180,0,0,47.5112,-122.257,175400.0,5,25,59,False
1,6414100192,2014-12-09,538000.0,3,2,2570,7242,2.0,0,0,...,2170,400,1991,47.7210,-122.319,225900.0,7,49,63,True
2,5631500400,2015-02-25,180000.0,2,1,770,10000,1.0,0,0,...,770,0,0,47.7379,-122.233,246600.0,9,81,82,False
3,2487200875,2014-12-09,604000.0,4,3,1960,5000,1.0,0,0,...,1050,910,0,47.5208,-122.393,280200.0,7,49,49,False
4,1954400510,2015-02-18,510000.0,3,2,1680,8080,1.0,0,0,...,1680,0,0,47.6168,-122.045,239850.0,9,81,28,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18559,263000018,2014-05-21,360000.0,3,2,1530,1131,3.0,0,0,...,1530,0,0,47.6993,-122.346,279200.0,0,0,5,False
18560,6600060120,2015-02-23,400000.0,4,2,2310,5813,2.0,0,0,...,2310,0,0,47.5107,-122.362,180100.0,9,81,1,False
18561,1523300141,2014-06-23,402101.0,2,1,1020,1350,2.0,0,0,...,1020,0,0,47.5944,-122.299,219400.0,1,1,5,False
18562,291310100,2015-01-16,400000.0,3,2,1600,2388,2.0,0,0,...,1600,0,0,47.5345,-122.069,308700.0,8,64,11,False


In [ ]:
def node_function(work_df, features, lambida, gama):
    
    H_node = work_df["hessian"].sum()
    G_node = work_df["gradient"].sum()

    if len(work_df) <= 100 :
        return {"type": "leaf" , "mean_target": work_df["target"].mean() }
    
    sorted_features = np.random.choice(features, 3)
    
    gain = -float("inf")
    limiar = None
    top_feature = None

    for i in sorted_features:
        temporary = work_df[i].unique()
        temporary.sort()
        limiars = [(temporary[l] + temporary[l+1])/2 for l in range(0,len(temporary)-1)]

        for j in limiars:
            left = work_df[work_df[i] <= j]
            right = work_df[work_df[i] > j]
            
            Gl = left["gradient"].sum()
            Hl = left["hessian"].sum()
            Gr = right["gradient"].sum()
            Hr = right["hessian"].sum()

            temporary1 = 0.5*(((Gl**2)/(Hl+lambida)) + ((Gr**2)/(Hr+lambida)) - ((G_node**2)/(H_node+lambida)))

            if temporary1 > gain: 
                gain = temporary1
                limiar = j
                top_feature = i

    if gain < gama:
        return {"type": "leaf" , "mean_target": work_df["target"].mean() }     
    
    if top_feature == None: 
        return {"type": "leaf" , "mean_target": work_df["target"].mean()}
    
    left = work_df[work_df[top_feature] <= limiar] 
    right =  work_df[work_df[top_feature] > limiar]

    if len(left) ==0 or len(right) ==0:
        return {"type": "leaf" , "mean_target": work_df["target"].mean()}  


    left_child = node_function(left, features, lambida, gama)
    right_child = node_function(right, features, lambida, gama)

    node = {"type": "node", "left_child": left_child, "right_child": right_child}

    return node



In [59]:
# terceira tentativa de implantar o XGBoos 
def XGBoost(data, features, target, lambida, gama):
    

    work = data[features].copy()
    work["target"] = data[target]

    media = data[target].mean()
    gradient = (-2)*(data[target] - media)
    hessian = len(data) *[2]
    
    work["gradient"] = gradient
    work["hessian"] = hessian

    
    result = node_function(work, features, lambida, gama)

    return result


In [14]:
features = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "sqft_above", "sqft_basement", "house_age"]
target = "price"

In [57]:
teste = XGBoost(df, features, "price", 1, 0)

18564


In [58]:
teste

{'type': 'node',
 'left_child': {'type': 'node',
  'left_child': {'type': 'node',
   'left_child': {'type': 'node',
    'left_child': {'type': 'node',
     'left_child': {'type': 'node',
      'left_child': {'type': 'node',
       'left_child': {'type': 'node',
        'left_child': {'type': 'node',
         'left_child': {'type': 'leaf', 'mean_target': 361837.1111111111},
         'right_child': {'type': 'node',
          'left_child': {'type': 'leaf', 'mean_target': 211735.48076923078},
          'right_child': {'type': 'node',
           'left_child': {'type': 'node',
            'left_child': {'type': 'leaf', 'mean_target': 320723.6666666667},
            'right_child': {'type': 'node',
             'left_child': {'type': 'leaf', 'mean_target': 264195.8245614035},
             'right_child': {'type': 'leaf', 'mean_target': 473000.0}}},
           'right_child': {'type': 'leaf', 'mean_target': 410387.5}}}},
        'right_child': {'type': 'node',
         'left_child': {'type': 'nod